In [ ]:
# Import libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# -----------------------------
# USE 3000 TRAIN SAMPLES ONLY
# -----------------------------
x_train = x_train[:3000]
y_train = y_train[:3000]

x_test = x_test[:1000]
y_test = y_test[:1000]

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# -----------------------------
# Load Pre-trained Model
# -----------------------------
base_model = keras.applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(32, 32, 3)
)

# Freeze all layers
for layer in base_model.layers:
    layer.trainable = False

# -----------------------------
# Add Classifier Head
# -----------------------------
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(64, activation='relu')(x)
output = layers.Dense(10, activation='softmax')(x)

model = keras.Model(inputs=base_model.input, outputs=output)

# Compile model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# -----------------------------
# TRAIN (FROZEN MODEL)
# -----------------------------
history_frozen = model.fit(
    x_train, y_train,
    epochs=2,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# -----------------------------
# FINE-TUNING
# -----------------------------
# Unfreeze last 10 layers
for layer in base_model.layers[-10:]:
    layer.trainable = True

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history_finetune = model.fit(
    x_train, y_train,
    epochs=2,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# -----------------------------
# SINGLE COMPARISON GRAPH
# -----------------------------
plt.plot(history_frozen.history['val_accuracy'], label='Frozen Model')
plt.plot(history_finetune.history['val_accuracy'], label='Fine-Tuned Model')

plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.title("Transfer Learning Comparison (3000 Samples)")

plt.legend()
plt.grid()
plt.show()

# -----------------------------
# EVALUATION
# -----------------------------
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)